# Stop delay dynamics: feasibility EDA

The future goal is a state-space dynamics model: **upstream-stop delay, current-stop delay, and a congestion proxy at time t** would inform **current-stop delay at t+1**. Actual delays and congestion measurements will eventually be supplied externally, and parameters could later be learned from those observations.

For now, this notebook only examines whether useful observations and patterns exist. It does **not** formulate equations, construct state-space matrices, fit parameters, or make predictions. The four-row figure compares speed, spatial delay change, and current/preceding-stop delays for one incoming stop segment.

Use the `transit_env` kernel. This notebook loads a trusted local cache offline; there are no downloads, database calls, or dataset writes.

In [ ]:
from pathlib import Path
import sys
from time import perf_counter

repo_root = next(
    (p for p in (Path.cwd().resolve(), Path.cwd().resolve().parent)
     if (p / 'forecasting' / 'forecasting').is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
sys.path.insert(0, str(repo_root / 'forecasting'))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from forecasting import get_backtest_dataset, prepare_stop_dynamics_eda

## Select the dataset and incoming segment

The default is **segment 4, direction 0**, Route `97-KEL`, in the two-month snapshot: **Hwy 97 at Westside Rd Stn (103745) -> Queensway Exchange Bay B (102853)**, shape `51737`. Segment IDs follow the existing congestion engine for the same dataset, route, and single-direction selection; a different snapshot or direction can give the same ID a different meaning. Check the resolved metadata below.

The notebook imports the checked-out forecasting package so it can run from either the repository root or this directory. Cache loading, data preparation, and plotting are separate: rerun only plotting cells to zoom without reloading or recomputing. The baseline always uses the complete selected snapshot, even when the display window is narrowed.

In [ ]:
backtest_dataset_id = '2026-11-10_2_months'
route_id = '97-KEL'
trip_direction_id = 0
segment_id = 6
time_bin_size = '5min'
display_timezone = 'America/Vancouver'

In [ ]:
load_started = perf_counter()
dataset = get_backtest_dataset(
    repo_root / 'data', source='cache', dataset_id=backtest_dataset_id,
)
static_dir = dataset.static_data_dir
stops = pd.read_csv(static_dir / 'stops.txt', usecols=['stop_id', 'stop_name'], dtype={'stop_id': 'string'})
stop_times = pd.read_csv(static_dir / 'stop_times.txt', usecols=['trip_id', 'stop_id', 'stop_sequence'], dtype={'trip_id': 'string', 'stop_id': 'string'})
trips = pd.read_csv(static_dir / 'trips.txt', usecols=['route_id', 'trip_id', 'direction_id', 'shape_id'], dtype={'route_id': 'string', 'trip_id': 'string', 'shape_id': 'string'})
load_seconds = perf_counter() - load_started
print(f'Dataset: {dataset.directory.name}; load and static read: {load_seconds:.2f} s')
print(f'Raw vehicle / trip rows: {len(dataset.vehicle_updates):,} / {len(dataset.trip_updates):,}')

## Prepare only the selected segment

`prepare_stop_dynamics_eda` builds a lightweight static catalogue with engine-compatible numbering, not route geometry. It matches speed probes by trip, direction, stop ID, and that trip's stop sequence; it selects latest stop predictions only for the matched endpoint visits.

Delay is the **latest reported arrival_delay**, selected by publication time, with database ID breaking ties. It is a proxy, not an independently measured arrival. Missing latest estimates are not replaced with older predictions or departure delays. Trip-start descriptors distinguish daily services; stop sequences distinguish repeated visits. Missing starts, unmatched GTFS visits, skipped/no-data/cancelled visits, unfinished arrivals, and unusable pairs are audited rather than guessed.

In [ ]:
prepare_started = perf_counter()
eda = prepare_stop_dynamics_eda(
    dataset.vehicle_updates, dataset.trip_updates, stops, stop_times, trips,
    route_id=route_id, trip_direction_id=trip_direction_id,
    segment_id=segment_id, time_bin_size=time_bin_size,
)
prepare_seconds = perf_counter() - prepare_started
current_visits = eda.stop_visits.query("exclusion_reason == 'included'")
upstream_visits = eda.upstream_visits.query("exclusion_reason == 'included'")
paired_visits = eda.delay_changes.query("exclusion_reason == 'included'")
display(eda.segment.to_frame('selected segment'))
print(f'Selected-segment preparation: {prepare_seconds:.2f} s')
print(f'Valid probes: {eda.speed_bins.probe_count.sum():,}; current visits: {len(current_visits):,}; upstream visits: {len(upstream_visits):,}; paired visits: {len(paired_visits):,}')
print(f'Engine median probe baseline: {eda.speed_bins.baseline_speed.iloc[0] * 3.6:.2f} km/h')

### Check coverage before interpreting patterns

The two-month cache's bundled GTFS begins on November 8, so some older realtime trip IDs are unmatched. Exclusions may be uneven over time. Inspect both stage-level reasons and daily UTC coverage; do not assume every day or bus is represented.

`candidate_probes` and `candidate_updates` count selected-stop publications before exact segment matching; `matched_*` count static matches, including rows subsequently excluded for missing starts or bad measurements. Their shares measure static matching, not measurement completeness. Usable current-stop visits, preceding-stop visits, and pairs are counted by their respective reported arrival day. Raw update counts are not counts of independent buses.

In [ ]:
display(eda.audit)
coverage_table = eda.coverage.copy()
coverage_table['probe_match_share'] = coverage_table.matched_probes.div(coverage_table.candidate_probes.where(coverage_table.candidate_probes.gt(0)))
coverage_table['update_match_share'] = coverage_table.matched_updates.div(coverage_table.candidate_updates.where(coverage_table.candidate_updates.gt(0)))
display(coverage_table)

## One incoming segment, four rows, one time axis

1. **Bus speed:** mean instantaneous vehicle-probe speed in each UTC bin, compared with the full-period median **individual probe** speed. This reproduces [Time-binned bus speed vs engine baseline](route_97_speed_congestion_analysis.ipynb), not completed-traversal speed. Repeated updates from one bus remain weighted as repeated probes.
2. **Spatial delay change:** for one trip instance and adjacent scheduled visits, **current arrival_delay - upstream arrival_delay**. Positive means added delay, negative means recovery. This is not a temporal difference between successive buses.
3. **Current-stop delay:** the current visit's final reported arrival-delay proxy. Valid current visits remain here even if upstream pairing fails. Positive means late; negative means early.
4. **Preceding-stop delay:** the upstream visit's final reported arrival-delay proxy, plotted at **its own upstream arrival time**, not the later current-stop arrival. Valid upstream visits remain here even if the current visit is missing or unusable. Standalone delay rows exclude arrivals before the recording window; upstream boundary evidence is still retained for paired delay changes.

Rows 2-4 show individual visits plus five-minute medians, not rolling averages over buses. Empty bins stay gaps; zero reference lines are not missing-data imputations. Speed is in km/h and delay in minutes. Hover reveals support and paired endpoint delays.

The shared clock does **not** establish simultaneous measurements: speed uses vehicle publication times; rows 2-3 use current-stop arrivals, and row 4 uses preceding-stop arrivals. The speed probes need not belong to the buses supplying delay points. Box zoom can change both axes in every row; the range selector/slider controls the shared time window without locking vertical zoom. The legend sits below the plot, separate from the title and time controls. Optional bounds below accept timezone-aware timestamps; naive values are interpreted as UTC.

In [ ]:
display_start = None
display_end = None

plot_started = perf_counter()
bin_times = eda.speed_bins.index.tz_convert(display_timezone)
figure = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.055,
    subplot_titles=('Bus speed vs engine baseline', 'Delay gained / recovered on the incoming segment', 'Delay at the current stop', 'Delay at the preceding stop'),
)

In [ ]:
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.speed_bins.observed_speed * 3.6,
    mode='lines+markers', name=f'Mean probe speed ({time_bin_size})',
    line=dict(color='royalblue'), connectgaps=False,
    customdata=eda.speed_bins[['probe_count', 'bus_count']],
    hovertemplate='%{x}<br>Speed: %{y:.2f} km/h<br>Probes: %{customdata[0]}<br>Distinct vehicles: %{customdata[1]}<extra>%{fullData.name}</extra>',
), row=1, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.speed_bins.baseline_speed * 3.6,
    mode='lines', name='Engine baseline (median probe speed)',
    line=dict(color='darkorange', dash='dash'),
), row=1, col=1);

In [ ]:
paired_hover = paired_visits.assign(
    upstream_minutes=paired_visits.upstream_arrival_delay / 60,
    current_minutes=paired_visits.arrival_delay / 60,
)[['trip_instance_key', 'from_stop_sequence', 'stop_sequence', 'upstream_minutes', 'current_minutes']]
figure.add_trace(go.Scattergl(
    x=paired_visits.arrival_time.dt.tz_convert(display_timezone),
    y=paired_visits.delta_delay_seconds / 60,
    mode='markers', name='Individual trip delay change',
    marker=dict(color='firebrick', size=5, opacity=0.45),
    customdata=paired_hover,
    hovertemplate='%{x}<br>Trip: %{customdata[0]}<br>Sequences: %{customdata[1]} -> %{customdata[2]}<br>Upstream: %{customdata[3]:.2f} min<br>Current: %{customdata[4]:.2f} min<br>Change: %{y:.2f} min<extra>%{fullData.name}</extra>',
), row=2, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.delay_bins.median_delta_delay_seconds / 60,
    mode='lines+markers', name=f'Median delay change ({time_bin_size})',
    line=dict(color='firebrick'), connectgaps=False,
    customdata=eda.delay_bins[['pair_count']],
    hovertemplate='%{x}<br>Median change: %{y:.2f} min<br>Paired visits: %{customdata[0]}<extra>%{fullData.name}</extra>',
), row=2, col=1);

In [ ]:
figure.add_trace(go.Scattergl(
    x=current_visits.arrival_time.dt.tz_convert(display_timezone),
    y=current_visits.arrival_delay / 60,
    mode='markers', name='Individual current-stop delay',
    marker=dict(color='seagreen', size=5, opacity=0.45),
    customdata=current_visits[['trip_instance_key', 'stop_sequence']],
    hovertemplate='%{x}<br>Trip: %{customdata[0]}<br>Stop sequence: %{customdata[1]}<br>Delay: %{y:.2f} min<extra>%{fullData.name}</extra>',
), row=3, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.delay_bins.median_current_delay_seconds / 60,
    mode='lines+markers', name=f'Median current-stop delay ({time_bin_size})',
    line=dict(color='seagreen'), connectgaps=False,
    customdata=eda.delay_bins[['visit_count']],
    hovertemplate='%{x}<br>Median delay: %{y:.2f} min<br>Current visits: %{customdata[0]}<extra>%{fullData.name}</extra>',
), row=3, col=1);

In [ ]:
figure.add_trace(go.Scattergl(
    x=upstream_visits.arrival_time.dt.tz_convert(display_timezone).tolist(),
    y=upstream_visits.arrival_delay / 60,
    mode='markers', name='Individual preceding-stop delay',
    marker=dict(color='mediumpurple', size=5, opacity=0.45),
    customdata=upstream_visits[['trip_instance_key', 'stop_sequence']],
    hovertemplate='%{x}<br>Trip: %{customdata[0]}<br>Stop sequence: %{customdata[1]}<br>Delay: %{y:.2f} min<extra>%{fullData.name}</extra>',
), row=4, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.delay_bins.median_upstream_delay_seconds / 60,
    mode='lines+markers', name=f'Median preceding-stop delay ({time_bin_size})',
    line=dict(color='mediumpurple'), connectgaps=False,
    customdata=eda.delay_bins[['upstream_visit_count']],
    hovertemplate='%{x}<br>Median delay: %{y:.2f} min<br>Preceding visits: %{customdata[0]}<extra>%{fullData.name}</extra>',
), row=4, col=1);

In [ ]:
figure.add_hline(y=0, line_dash='dot', line_color='gray', row=2, col=1)
figure.add_hline(y=0, line_dash='dot', line_color='gray', row=3, col=1)
figure.add_hline(y=0, line_dash='dot', line_color='gray', row=4, col=1)
figure.update_yaxes(title_text='Speed (km/h)', rangemode='tozero', row=1, col=1)
figure.update_yaxes(title_text='Delay change (min)', row=2, col=1)
figure.update_yaxes(title_text='Current delay (min)', row=3, col=1)
figure.update_yaxes(title_text='Preceding delay (min)', row=4, col=1)
figure.update_yaxes(fixedrange=False)
view_start = pd.to_datetime(display_start, utc=True) if display_start is not None else eda.speed_bins.index[0]
view_end = pd.to_datetime(display_end, utc=True) if display_end is not None else eda.speed_bins.index[-1] + pd.Timedelta(time_bin_size)
if pd.isna(view_start) or pd.isna(view_end) or view_start >= view_end:
    raise ValueError('Display bounds must be valid timestamps with start before end.')
figure.update_xaxes(range=[view_start.tz_convert(display_timezone), view_end.tz_convert(display_timezone)])
figure.update_xaxes(
    title_text=f'Time ({display_timezone})', row=4, col=1,
    rangeslider=dict(visible=True, thickness=0.06),
    rangeselector=dict(buttons=[
        dict(count=7, label='7d', step='day', stepmode='backward'),
        dict(count=1, label='1m', step='month', stepmode='backward'),
        dict(label='All', step='all'),
    ]),
)
if paired_visits.empty:
    figure.add_annotation(text='No usable paired delays: inspect audit', x=0.5, y=0.5, xref='x2 domain', yref='y2 domain', showarrow=False)
if current_visits.empty:
    figure.add_annotation(text='No usable current-stop delays: inspect audit', x=0.5, y=0.5, xref='x3 domain', yref='y3 domain', showarrow=False)
if upstream_visits.empty:
    figure.add_annotation(text='No usable preceding-stop delays: inspect audit', x=0.5, y=0.5, xref='x4 domain', yref='y4 domain', showarrow=False)
figure.update_layout(
    title=f"Segment {segment_id}, direction {trip_direction_id}, shape {eda.segment['shape_id']}<br>{eda.segment['from_stop_name']} -> {eda.segment['to_stop_name']}",
    height=1300, hovermode='x unified', hoversubplots='axis',
    legend=dict(orientation='h', x=0, xanchor='left', y=-0.18, yanchor='top'),
    margin=dict(t=140, b=300),
)
plot_seconds = perf_counter() - plot_started
print(f'Figure construction: {plot_seconds:.2f} s (browser rendering is separate)')
figure.show()

## What this EDA can tell us

- Do speed bins and paired delays overlap often enough in time to support further study? Are the patterns consistent across well-supported days rather than driven by one bus or sparse bins?
- Do low-speed periods coincide with delay gains, or do buses remain late without adding delay on this segment? Row 2 separates accumulated lateness from changes on the incoming segment; rows 3-4 show the current and preceding stop's total state-like delay proxies at their respective passage times.
- How much evidence is missing because of historical GTFS mismatch, missing trip starts, or incomplete endpoints? A missing bin is unknown, not uncongested.

**Limits:** arrival-to-arrival changes include upstream dwell, holding, and schedule slack, so they are not pure congestion measurements. The full-period median speed baseline and final reported delays can use information unavailable at an earlier prediction time. This is retrospective EDA, not a leakage-free backtest or proof of causality, forecast skill, or state-space parameter identifiability. Same-trip upstream/current arrival delays also refer to different passage times, not simultaneous stop states at t.

Before future modeling, define the time grid and external measurement semantics, distinguish simultaneous stop states from successive bus visits, establish point-in-time availability, and validate against later held-out data. None of those modeling steps is performed here.